In [ ]:
# Example 1: Basic Classification Metrics and Model Evaluation
import evaluate
import numpy as np
from sklearn.metrics import confusion_matrix

print("Basic classification metrics and model evaluation:")
print("=" * 49)

# Simulate classification predictions and ground truth
np.random.seed(42)
y_true = np.random.randint(0, 3, 100)  # 3-class classification
y_pred = y_true.copy()
# Add some noise to create realistic prediction errors
noise_indices = np.random.choice(100, 20, replace=False)
y_pred[noise_indices] = np.random.randint(0, 3, 20)

print(f"Dataset: 100 samples, 3 classes")
print(f"True distribution: {np.bincount(y_true)}")
print(f"Pred distribution: {np.bincount(y_pred)}")

# Load basic classification metrics
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

print(f"\n1. Basic classification metrics:")
acc_score = accuracy.compute(predictions=y_pred, references=y_true)
prec_score = precision.compute(predictions=y_pred, references=y_true, average='weighted')
rec_score = recall.compute(predictions=y_pred, references=y_true, average='weighted')
f1_score = f1.compute(predictions=y_pred, references=y_true, average='weighted')

print(f"   Accuracy: {acc_score['accuracy']:.3f}")
print(f"   Precision (weighted): {prec_score['precision']:.3f}")
print(f"   Recall (weighted): {rec_score['recall']:.3f}")
print(f"   F1-score (weighted): {f1_score['f1']:.3f}")

# VALUE CHECK: sklearn recomputes the same four quantities from the same deterministic inputs
# (np.random.seed(42) above). It is a plumbing + value check rather than an independent
# reimplementation -- HF's accuracy/precision/recall/f1 module scripts call sklearn themselves -- but
# it still fails on a wrong kwarg, a wrong result key, a wrong averaging mode or a broken aggregation.
from sklearn.metrics import accuracy_score as sk_accuracy, f1_score as sk_f1
from sklearn.metrics import precision_score as sk_precision, recall_score as sk_recall
assert abs(acc_score['accuracy'] - sk_accuracy(y_true, y_pred)) < 1e-12, acc_score
assert abs(prec_score['precision'] - sk_precision(y_true, y_pred, average='weighted')) < 1e-12, prec_score
assert abs(rec_score['recall'] - sk_recall(y_true, y_pred, average='weighted')) < 1e-12, rec_score
assert abs(f1_score['f1'] - sk_f1(y_true, y_pred, average='weighted')) < 1e-12, f1_score

# Detailed per-class analysis
print(f"\n2. Per-class analysis:")
prec_per_class = precision.compute(predictions=y_pred, references=y_true, average=None)
rec_per_class = recall.compute(predictions=y_pred, references=y_true, average=None)
f1_per_class = f1.compute(predictions=y_pred, references=y_true, average=None)

for i in range(3):
    print(f"   Class {i}: P={prec_per_class['precision'][i]:.3f}, "
          f"R={rec_per_class['recall'][i]:.3f}, F1={f1_per_class['f1'][i]:.3f}")

# Multi-metric evaluation - compute metrics individually
# Note: evaluate.combine() doesn't allow passing metric-specific parameters,
# so we compute each metric separately for multiclass problems
print(f"\n3. Combined metric evaluation:")

# Compute accuracy (no average parameter needed)
acc_result = accuracy.compute(predictions=y_pred, references=y_true)

# Compute other metrics with macro averaging
prec_macro = precision.compute(predictions=y_pred, references=y_true, average='macro')
rec_macro = recall.compute(predictions=y_pred, references=y_true, average='macro')
f1_macro = f1.compute(predictions=y_pred, references=y_true, average='macro')

# Aggregate results
combined_results = {
    'accuracy': acc_result['accuracy'],
    'precision_macro': prec_macro['precision'],
    'recall_macro': rec_macro['recall'],
    'f1_macro': f1_macro['f1']
}

print(f"   Accuracy: {combined_results['accuracy']:.3f}")
print(f"   Precision (macro): {combined_results['precision_macro']:.3f}")
print(f"   Recall (macro): {combined_results['recall_macro']:.3f}")
print(f"   F1-score (macro): {combined_results['f1_macro']:.3f}")

# Additional metrics - confusion matrix analysis
print(f"\n4. Confusion matrix analysis:")
cm = confusion_matrix(y_true, y_pred)
print(f"   Confusion matrix:")
for i, row in enumerate(cm):
    print(f"     Class {i}: {row}")

# Per-class accuracy from confusion matrix
class_accuracies = cm.diagonal() / cm.sum(axis=1)
print(f"   Per-class accuracy:")
for i, acc in enumerate(class_accuracies):
    print(f"     Class {i}: {acc:.3f}")

print("\nBasic classification metrics evaluation completed successfully!")

In [ ]:
# Example 2: Natural Language Processing evaluation metrics
import evaluate

print("Natural Language Processing evaluation metrics:")
print("=" * 44)

_REFERENCES = [
    "The cat sat on the mat.",
    "Machine learning is transforming technology.",
    "Climate change requires immediate action.",
]
_NEAR = [  # a paraphrase of each reference: must score above nonsense and below an exact copy
    "A cat was sitting on the mat.",
    "ML is revolutionizing tech industry.",
    "Climate change needs urgent response.",
]
_DISJOINT = [  # shares essentially no content with the references
    "Completely unrelated text here.",
    "Nothing at all in common.",
    "Zebra xylophone quartz.",
]

# 1. BLEU — n-gram precision for translation. Identity is the anchor: a correct BLEU gives exactly
#    1.0 with every n-gram precision at 1.0.
bleu = evaluate.load("bleu")
_bleu_identity = bleu.compute(predictions=_REFERENCES, references=[[r] for r in _REFERENCES])
_bleu_near = bleu.compute(predictions=_NEAR, references=[[r] for r in _REFERENCES])
assert _bleu_identity["bleu"] == 1.0, f"BLEU of a text against itself must be 1.0, got {_bleu_identity['bleu']}"
assert _bleu_identity["precisions"] == [1.0, 1.0, 1.0, 1.0], (
    f"BLEU identity must give precision 1.0 at every n-gram order, got {_bleu_identity['precisions']}"
)
assert 0.0 < _bleu_near["bleu"] < _bleu_identity["bleu"], (
    f"BLEU of a paraphrase must land strictly between 0 and the identity score, got {_bleu_near['bleu']}"
)
assert len(_bleu_near["precisions"]) == 4, (
    f"BLEU must report four n-gram precisions, got {len(_bleu_near['precisions'])}"
)
print(f"1. BLEU  identity={_bleu_identity['bleu']:.4f}  paraphrase={_bleu_near['bleu']:.4f}")

# 2. Google BLEU and METEOR — the three-arm ordering is the whole assertion. Both are computed on
#    identical / near / disjoint inputs and must rank them in that order.
#    meteor needs `pip install nltk`; it downloads NLTK corpora on first use.
for _name, _metric_kwargs in (("google_bleu", "bracketed"), ("meteor", "flat")):
    _metric = evaluate.load(_name)

    def _score(predictions, _metric=_metric, _shape=_metric_kwargs):
        _refs = [[r] for r in _REFERENCES] if _shape == "bracketed" else _REFERENCES
        return float(_metric.compute(predictions=predictions, references=_refs)[_name])

    _identity, _near, _disjoint = _score(_REFERENCES), _score(_NEAR), _score(_DISJOINT)
    assert _identity > _near > _disjoint >= 0.0, (
        f"{_name} must rank identical > paraphrase > disjoint; got "
        f"identity={_identity:.4f} near={_near:.4f} disjoint={_disjoint:.4f}"
    )
    assert _identity > 0.99, f"{_name} of a text against itself must be ~1.0, got {_identity:.4f}"
    print(f"2. {_name:12s} identity={_identity:.4f}  paraphrase={_near:.4f}  disjoint={_disjoint:.4f}")

# 3. exact_match — a pure string comparison, so both ends are exactly known: identical strings
#    score 1.0 and paraphrases score 0.0, and both ends are asserted.
exact_match = evaluate.load("exact_match")
_em_identity = float(exact_match.compute(predictions=_REFERENCES, references=_REFERENCES)["exact_match"])
_em_near = float(exact_match.compute(predictions=_NEAR, references=_REFERENCES)["exact_match"])
assert _em_identity == 1.0, f"exact_match of identical strings must be 1.0, got {_em_identity}"
assert _em_near == 0.0, f"exact_match of three paraphrases must be 0.0, got {_em_near}"
print(f"3. exact_match  identical={_em_identity}  paraphrase={_em_near}")

# 4. Token-level classification, checked with accuracy + f1 (the seqeval NER metric needs an extra
#    package) on a hand-built confusion: predictions [0,1,1,0] against references [0,1,0,0] is 3 of 4
#    correct (0.75) with one false positive and no false negative, giving precision 1/2, recall 1/1
#    and F1 exactly 2/3. Both values are asserted exactly.
_PREDICTED_LABELS = [0, 1, 1, 0]
_TRUE_LABELS = [0, 1, 0, 0]
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
_accuracy = float(accuracy.compute(predictions=_PREDICTED_LABELS, references=_TRUE_LABELS)["accuracy"])
_f1 = float(f1.compute(predictions=_PREDICTED_LABELS, references=_TRUE_LABELS)["f1"])
assert _accuracy == 0.75, f"3 of 4 labels correct must give accuracy 0.75, got {_accuracy}"
assert abs(_f1 - 2.0 / 3.0) < 1e-12, f"precision 1/2 with recall 1/1 must give F1 2/3, got {_f1}"
print(f"4. accuracy={_accuracy}  f1={_f1:.4f}")

print("\nNLP evaluation metrics completed successfully!")

In [ ]:
# Example 3: Regression and Statistical Evaluation Metrics
import evaluate
import numpy as np

print("Regression and statistical evaluation metrics:")
print("=" * 41)

# Generate synthetic regression data
np.random.seed(42)
n_samples = 200
x = np.linspace(0, 10, n_samples)
y_true = 2.5 * x + 1.3 + np.random.normal(0, 0.8, n_samples)  # Linear with noise
y_pred = 2.3 * x + 1.5 + np.random.normal(0, 0.5, n_samples)  # Slightly different model

print(f"Dataset: {n_samples} samples")
print(f"True range: [{y_true.min():.2f}, {y_true.max():.2f}]")
print(f"Pred range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")

# Basic regression metrics
print(f"\n1. Basic regression metrics:")

# Mean Absolute Error
mae = evaluate.load("mae")
mae_score = mae.compute(predictions=y_pred, references=y_true)
print(f"   Mean Absolute Error: {mae_score['mae']:.3f}")

# Mean Squared Error
mse = evaluate.load("mse")
mse_score = mse.compute(predictions=y_pred, references=y_true)
print(f"   Mean Squared Error: {mse_score['mse']:.3f}")
print(f"   Root Mean Squared Error: {np.sqrt(mse_score['mse']):.3f}")

# Mean Absolute Percentage Error
mape = evaluate.load("mape")
mape_score = mape.compute(predictions=y_pred, references=y_true)
print(f"   Mean Absolute Percentage Error: {mape_score['mape']:.3f}%")

# R-squared coefficient
try:
    r_squared = evaluate.load("r_squared")
    r2_score = r_squared.compute(predictions=y_pred, references=y_true)
    print(f"   R-squared: {r2_score['r_squared']:.3f}")
except:
    from sklearn.metrics import r2_score
    r2 = r2_score(y_true, y_pred)
    print(f"   R-squared (sklearn): {r2:.3f}")

# Statistical analysis
print(f"\n2. Statistical analysis:")
residuals = y_true - y_pred
print(f"   Residual statistics:")
print(f"     Mean: {np.mean(residuals):.4f}")
print(f"     Std: {np.std(residuals):.4f}")
print(f"     Min: {np.min(residuals):.3f}")
print(f"     Max: {np.max(residuals):.3f}")

# Pearson correlation
correlation = np.corrcoef(y_true, y_pred)[0, 1]
print(f"   Pearson correlation: {correlation:.4f}")

# Classification-style evaluation for regression (binning)
print(f"\n3. Categorical evaluation of regression:")
# Convert regression to classification problem by binning
n_bins = 5
true_bins = np.digitize(y_true, np.linspace(y_true.min(), y_true.max(), n_bins))
pred_bins = np.digitize(y_pred, np.linspace(y_pred.min(), y_pred.max(), n_bins))

# Compute accuracy for binned predictions
binned_accuracy = evaluate.load("accuracy")
bin_acc_score = binned_accuracy.compute(predictions=pred_bins, references=true_bins)
print(f"   Binned accuracy ({n_bins} bins): {bin_acc_score['accuracy']:.3f}")

# F1 score for binned predictions
binned_f1 = evaluate.load("f1")
bin_f1_score = binned_f1.compute(predictions=pred_bins, references=true_bins, average='weighted')
print(f"   Binned F1-score: {bin_f1_score['f1']:.3f}")

print("\nRegression and statistical evaluation completed successfully!")

In [ ]:
# Example 4: Custom Metrics and Advanced Evaluation Patterns
import evaluate
import numpy as np
from typing import List, Dict, Any
import datasets

print("Custom metrics and advanced evaluation patterns:")
print("=" * 45)

# Create custom evaluation metric
print("1. Custom metric creation:")

class CustomMeanBias(evaluate.Metric):
    def _info(self):
        return evaluate.MetricInfo(
            description="Custom metric that calculates mean bias and absolute bias",
            citation="Custom implementation for demonstration",
            features=datasets.Features({
                'predictions': datasets.Value('float'),
                'references': datasets.Value('float'),
            }),
        )
    
    def _compute(self, predictions, references):
        predictions = np.array(predictions)
        references = np.array(references)
        
        bias = np.mean(predictions - references)
        abs_bias = np.mean(np.abs(predictions - references))
        rel_bias = bias / np.mean(references) if np.mean(references) != 0 else 0
        
        return {
            "mean_bias": float(bias),
            "absolute_bias": float(abs_bias),
            "relative_bias": float(rel_bias)
        }

# Test custom metric
custom_metric = CustomMeanBias()
y_true = [1.0, 2.5, 3.2, 4.1, 5.0]
y_pred = [1.1, 2.3, 3.5, 4.0, 5.2]

custom_results = custom_metric.compute(predictions=y_pred, references=y_true)
print(f"   Custom metric results: {custom_results}")

# Batch evaluation with multiple metrics
print(f"\n2. Batch evaluation with multiple datasets:")

datasets_dict = {
    'dataset_A': ([1, 2, 3, 4, 5], [1.1, 1.9, 3.2, 3.8, 5.1]),
    'dataset_B': ([2, 4, 6, 8, 10], [2.2, 3.8, 6.1, 7.9, 10.2]),
    'dataset_C': ([0.5, 1.5, 2.5], [0.6, 1.4, 2.6])
}

metrics_suite = ["mae", "mse"]
evaluation_results = {}

for dataset_name, (true_vals, pred_vals) in datasets_dict.items():
    dataset_results = {}
    
    for metric_name in metrics_suite:
        metric = evaluate.load(metric_name)
        score = metric.compute(predictions=pred_vals, references=true_vals)
        dataset_results[metric_name] = score[metric_name]
    
    evaluation_results[dataset_name] = dataset_results
    print(f"   {dataset_name}: MAE={dataset_results['mae']:.3f}, MSE={dataset_results['mse']:.3f}")

# Statistical significance testing
print(f"\n3. Statistical significance and confidence intervals:")

def bootstrap_metric(predictions, references, metric_func, n_bootstrap=1000):
    """Calculate confidence intervals using bootstrap sampling"""
    n_samples = len(predictions)
    bootstrap_scores = []
    
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_samples, n_samples, replace=True)
        boot_pred = [predictions[i] for i in indices]
        boot_ref = [references[i] for i in indices]
        score = metric_func.compute(predictions=boot_pred, references=boot_ref)
        bootstrap_scores.append(list(score.values())[0])
    
    mean_score = np.mean(bootstrap_scores)
    ci_lower = np.percentile(bootstrap_scores, 2.5)
    ci_upper = np.percentile(bootstrap_scores, 97.5)
    
    return mean_score, ci_lower, ci_upper

# Test bootstrap confidence intervals
mae_metric = evaluate.load("mae")
mean_mae, ci_low, ci_high = bootstrap_metric(y_pred, y_true, mae_metric, n_bootstrap=100)
print(f"   MAE: {mean_mae:.3f} (95% CI: [{ci_low:.3f}, {ci_high:.3f}])")

print("\nCustom metrics and advanced evaluation completed successfully!")

In [ ]:
# Example 5: Production Model Evaluation Pipeline
import evaluate
import numpy as np
from typing import Dict, List, Any
import time

print("Production model evaluation pipeline:")
print("=" * 37)

# Production evaluation framework
class ModelEvaluationPipeline:
    def __init__(self, task_type: str = "classification"):
        self.task_type = task_type
        self.metrics_cache = {}
        self.evaluation_history = []
        
        # Define metric suites for different tasks
        self.metric_suites = {
            "classification": ["accuracy", "precision", "recall", "f1"],
            "regression": ["mae", "mse", "mape"],
            "text_generation": ["bleu", "rouge"],
        }
    
    def load_metrics(self, custom_metrics: List[str] = None):
        """Load and cache evaluation metrics"""
        metrics_to_load = custom_metrics or self.metric_suites.get(self.task_type, ["accuracy"])
        
        for metric_name in metrics_to_load:
            if metric_name not in self.metrics_cache:
                self.metrics_cache[metric_name] = evaluate.load(metric_name)
        
        print(f"   Loaded {len(self.metrics_cache)} metrics: {list(self.metrics_cache.keys())}")
    
    def evaluate_model(self, predictions, references, model_name: str = "default"):
        """Comprehensive model evaluation"""
        start_time = time.time()
        results = {"model": model_name, "timestamp": start_time}
        
        # Compute all loaded metrics
        for metric_name, metric in self.metrics_cache.items():
            if metric_name in ["precision", "recall", "f1"] and self.task_type == "classification":
                # Multi-class metrics
                macro_score = metric.compute(predictions=predictions, references=references, average='macro')
                weighted_score = metric.compute(predictions=predictions, references=references, average='weighted')
                results[f"{metric_name}_macro"] = macro_score[metric_name]
                results[f"{metric_name}_weighted"] = weighted_score[metric_name]
            else:
                score = metric.compute(predictions=predictions, references=references)
                results.update(score)
        
        
        # Add evaluation metadata
        results["evaluation_time"] = time.time() - start_time
        results["n_samples"] = len(predictions)
        
        # Store evaluation history
        self.evaluation_history.append(results)
        return results

# Initialize production pipeline
print("1. Production evaluation pipeline setup:")
pipeline = ModelEvaluationPipeline(task_type="classification")
pipeline.load_metrics()
assert sorted(pipeline.metrics_cache) == ["accuracy", "f1", "precision", "recall"], sorted(pipeline.metrics_cache)

# Simulate model evaluation scenario
print(f"\n2. Multi-model comparison:")
np.random.seed(42)

# Generate synthetic data for 3 different models
n_samples = 1000
y_true = np.random.randint(0, 4, n_samples)  # 4-class classification

models = {
    "model_baseline": y_true + np.random.randint(-1, 2, n_samples),  # Poor model
    "model_improved": y_true.copy(),  # Perfect model
    "model_production": y_true + (np.random.random(n_samples) < 0.1) * np.random.randint(-2, 3, n_samples)  # Good model
}

# Clip predictions to valid range
for model_name in models:
    models[model_name] = np.clip(models[model_name], 0, 3)

# Evaluate all models
for model_name, y_pred in models.items():
    results = pipeline.evaluate_model(y_pred, y_true, model_name)
    
    print(f"   {model_name}:")
    print(f"     Accuracy: {results.get('accuracy', 'N/A'):.3f}")
    print(f"     F1 (weighted): {results.get('f1_weighted', 'N/A'):.3f}")
    print(f"     Evaluation time: {results['evaluation_time']:.4f}s")

# expected (np.random.seed(42), evaluate 0.4.6): every metric computes, and each model scores where its noise puts it
expected_scores = {"model_baseline": (0.509, 0.5141603905822756), "model_improved": (1.0, 1.0), "model_production": (0.94, 0.9398644492716076)}
for history_entry in pipeline.evaluation_history:
    want_accuracy, want_f1 = expected_scores[history_entry["model"]]
    assert abs(history_entry["accuracy"] - want_accuracy) < 1e-9 and abs(history_entry["f1_weighted"] - want_f1) < 1e-9, history_entry
    assert {"precision_macro", "precision_weighted", "recall_macro", "recall_weighted", "f1_macro"} <= history_entry.keys(), sorted(history_entry)

# Model comparison and reporting
print(f"\n3. Automated model comparison:")
best_model = None
best_score = -1

for eval_result in pipeline.evaluation_history:
    current_score = eval_result.get('f1_weighted', 0)
    if current_score > best_score:
        best_score = current_score
        best_model = eval_result['model']

print(f"   Best model: {best_model} (F1-weighted: {best_score:.3f})")
assert best_model == "model_improved", best_model

# Performance monitoring
print(f"\n4. Performance monitoring and alerts:")
performance_threshold = 0.85
models_below_threshold = [
    result['model'] for result in pipeline.evaluation_history 
    if result.get('accuracy', 0) < performance_threshold
]

if models_below_threshold:
    print(f"   ⚠️  Models below threshold ({performance_threshold}): {models_below_threshold}")
else:
    print(f"   ✅ All models meet performance threshold ({performance_threshold})")

# Evaluation summary statistics
print(f"\n5. Evaluation summary:")
total_evaluations = len(pipeline.evaluation_history)
total_time = sum(r['evaluation_time'] for r in pipeline.evaluation_history)
avg_time = total_time / total_evaluations if total_evaluations > 0 else 0

print(f"   Total evaluations: {total_evaluations}")
print(f"   Total evaluation time: {total_time:.4f}s")
print(f"   Average evaluation time: {avg_time:.4f}s per model")
print(f"   Samples evaluated: {pipeline.evaluation_history[0]['n_samples'] * total_evaluations}")

print("\nProduction model evaluation pipeline completed successfully!")